# Full platform demo

End-to-end run of the urban data platform:

1. Download raw data and bronze-ingest
2. Silver promote + gold integrate (validate along the way)
3. Run analytical queries Q1–Q6
4. Spark query optimizations (cache, partition pruning, broadcast, AQE)
5. Build gold data products
6. Benchmark products vs on-demand / AQE
7. Generate incremental updates (schema evolution) and re-ingest
8. Show monitoring (`pipeline_runs`)

Prefer the project `.venv` Jupyter kernel. Set flags in the setup cell as needed.

## 0. Setup

In [10]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, ensure_runtime

ensure_runtime()

PosixPath('/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs')

## 1. Download raw data and bronze ingest

In [11]:
from src.bronze.download import download_raw
from src.bronze.ingest import ingest_all
from src.lake import RAW

download_raw()

spark = create_spark("run-pipeline")
print("\n=== BRONZE ingest ===")
ingest_all(spark)

Retrieving folder contents


Processing file 1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW air_quality.zip
Processing file 1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t taxi_zone_lookup.csv
Processing file 1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M weather.csv
Processing file 17v0eFEontYEKtGoqyB0v9rj_BEDc7snA yellow_tripdata_2024-01.parquet
Processing file 1N-dRuGdd_lOYGAbdbgJJWMyIsV_lz057 yellow_tripdata_2024-02.parquet
Processing file 1oUxC0cLWqOatddyT8aB06VvFFZY0-lu2 yellow_tripdata_2024-03.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW
From (redirected): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW&confirm=t&uuid=48d33f2f-b660-47a4-995d-c0fd900326b7
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/air_quality.zip
100%|██████████| 66.3M/66.3M [00:02<00:00, 27.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/taxi_zone_lookup.csv
100%|██████████| 12.3k/12.3k [00:00<00:00, 4.78MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/weather.csv
100%|██████████| 1.07M/1.07M [00:00<00:00, 22.4MB/s]
Downloading...
From: https:

unzip data/drive_download/air_quality.zip -> data/raw/air_quality
copy  yellow_tripdata_2024-03.parquet -> data/raw/taxi_trips/yellow
copy  taxi_zone_lookup.csv -> data/raw/taxi_zones
copy  weather.csv -> data/raw/weather
copy  yellow_tripdata_2024-02.parquet -> data/raw/taxi_trips/yellow
copy  yellow_tripdata_2024-01.parquet -> data/raw/taxi_trips/yellow


26/09/24 21:51:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/24 21:51:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


...

[air_quality] schema ok=True


[bronze/air_quality] written (117,438 rows)


[taxi_trips/yellow] schema ok=True


[bronze/taxi_trips] written (9,554,778 rows)
[taxi_zones] schema ok=True
[bronze/taxi_zones] written (265 rows)
[weather] schema ok=True


[bronze/weather] written (8,784 rows)


## 2. Silver promote + gold integrate

In [12]:
from src.silver.promote import promote_all
from src.gold.integrate import integrate
from src.benchmark.storage import compare_integrated_layouts
from src.lake import GOLD, SILVER, read_delta, show_delta

print("=== SILVER ===")
promote_all(spark)

print("\nSilver tables")
for name in ["taxi_zones", "weather", "air_quality", "taxi_trips"]:
    show_delta(spark, SILVER / name, n=1)

print("\n=== GOLD integrate ===")
integrate(spark)
compare_integrated_layouts()

=== SILVER ===


[silver/air_quality] kept=112,838  rejected=4,600


[silver/taxi_trips] kept=9,417,383  rejected=137,395
[silver/taxi_zones] kept=265  rejected=0


[silver/weather] kept=8,784  rejected=0

Silver tables
taxi_zones: 265 rows @ data/lake/silver/taxi_zones
+-----------+-------+--------------+------------+
|location_id|borough|zone          |service_zone|
+-----------+-------+--------------+------------+
|1          |EWR    |Newark Airport|EWR         |
+-----------+-------+--------------+------------+
only showing top 1 row

weather: 8784 rows @ data/lake/silver/weather
+-----------+-------------------+-------------+-----------------+----------------+----------------+
|station_id |obs_timestamp      |temperature_c|wind_speed_ms    |observation_date|observation_hour|
+-----------+-------------------+-------------+-----------------+----------------+----------------+
|72505394728|2024-01-03 00:00:00|5.6          |5.694444444444445|2024-01-03      |0               |
+-----------+-------------------+-------------+-----------------+----------------+----------------+
only showing top 1 row

air_quality: 112838 rows @ data/lake/silver/air_qu

write by_date: 7.7s
integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+---------------------+---------------+-----------+----------+------------+------------+-------------+-----------------+----+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone          |pickup_borough|dropoff_location_id|dropoff_zone         |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25|pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+---------------------+---------------+-----------+

write by_borough: 6.7s
Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


## 3. Analytical queries Q1–Q6

In [4]:
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND,
    QUERY_2_WEATHER_DISTANCE,
    QUERY_3_PM25_DEMAND,
    QUERY_4_ZONE_WEATHER_SENSITIVITY,
    QUERY_5_PEAK_HOURS_BY_DOW,
    QUERY_6_MONTHLY_TRENDS,
)

read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)

for name, sql in [
    ("Q1 monthly zone demand", QUERY_1_MONTHLY_ZONE_DEMAND),
    ("Q2 weather distance", QUERY_2_WEATHER_DISTANCE),
    ("Q3 pm25 demand", QUERY_3_PM25_DEMAND),
    ("Q4 zone weather sensitivity", QUERY_4_ZONE_WEATHER_SENSITIVITY),
    ("Q5 peak hours by DOW", QUERY_5_PEAK_HOURS_BY_DOW),
    ("Q6 monthly trends", QUERY_6_MONTHLY_TRENDS),
]:
    print(f"\n=== {name} ===")
    spark.sql(sql).show(15, truncate=False)


=== Q1 monthly zone demand ===


+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|trip_month|pickup_location_id|pickup_borough|pickup_zone                 |total_trips|active_days|avg_daily_trips|
+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|2024-01-01|161               |Manhattan     |Midtown Center              |141738     |31         |4572.19        |
|2024-01-01|237               |Manhattan     |Upper East Side South       |141263     |31         |4556.87        |
|2024-01-01|132               |Queens        |JFK Airport                 |141159     |31         |4553.52        |
|2024-01-01|236               |Manhattan     |Upper East Side North       |135334     |31         |4365.61        |
|2024-01-01|162               |Manhattan     |Midtown East                |105466     |31         |3402.13        |
|2024-01-01|230               |Manhattan     |Times Sq/Theatre District 

+----------+-----+--------------+--------------+
|pm25_level|trips|observed_hours|trips_per_hour|
+----------+-----+--------------+--------------+
|34.0      |4188 |1             |4188.0        |
|33.0      |9071 |2             |4535.5        |
|31.0      |5019 |1             |5019.0        |
|30.0      |18438|3             |6146.0        |
|29.0      |18240|2             |9120.0        |
|28.0      |17702|3             |5900.67       |
|27.0      |14835|2             |7417.5        |
|26.0      |66047|10            |6604.7        |
|25.0      |18435|3             |6145.0        |
|24.0      |54765|10            |5476.5        |
|23.0      |20069|5             |4013.8        |
|22.0      |54775|12            |4564.58       |
|21.0      |80355|17            |4726.76       |
|20.0      |83362|16            |5210.13       |
|19.0      |79234|17            |4660.82       |
+----------+-----+--------------+--------------+
only showing top 15 rows


=== Q4 zone weather sensitivity ===


+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.25  |13.0  |12.66 |10.69  |42.28        |
|Financial District North     |26.62  |21.65 |21.07 |18.24  |38.27        |
|Meatpacking/West Village West|49.81  |38.38 |40.34 |34.71  |37.0         |
|Lower East Side              |58.06  |45.25 |48.38 |41.47  |34.35        |
|Little Italy/NoLiTa          |52.69  |41.92 |43.93 |38.29  |32.57        |
|Greenwich Village South      |75.86  |62.62 |63.19 |57.27  |28.72        |
|West Village                 |121.51 |101.08|101.63|92.03  |28.33        |
|Battery Park City            |31.51  |28.26 |26.96 |23.97  |27.24        |
|TriBeCa/Civic Center         |66.51  |58.87 |57.36 |50.75  |27.0         |
|World Trade Center           |25.62  |21.19 |21.69 |19.84  |26.17        |
|East Villag

26/09/24 18:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+---------+----------+-----------+---------+
|day_of_week|peak_hour|trip_count|active_days|avg_trips|
+-----------+---------+----------+-----------+---------+
|Monday     |18       |80482     |13         |6190.92  |
|Tuesday    |18       |99614     |13         |7662.62  |
|Wednesday  |18       |110914    |13         |8531.85  |
|Thursday   |18       |119124    |13         |9163.38  |
|Friday     |18       |102921    |13         |7917.0   |
|Saturday   |19       |96485     |13         |7421.92  |
|Sunday     |0        |79601     |13         |6123.15  |
+-----------+---------+----------+-----------+---------+


=== Q6 monthly trends ===
+----------+-----------+-----------+---------------+--------------------+------------------------+
|trip_month|total_trips|active_days|avg_daily_trips|prev_month_avg_daily|pct_change_vs_prev_month|
+----------+-----------+-----------+---------------+--------------------+------------------------+
|2024-01-01|2926910    |31         |94416.45    

26/09/24 18:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:09:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Query optimizations (on silver)

Cache reuse, partition pruning, broadcast joins, and AQE — same experiments as `query_optimization.ipynb`.

In [5]:
from pyspark.sql.functions import avg, broadcast


def benchmark_and_verify(df_baseline, df_optimized, name="Optimization Test"):
    print(f"=== {name} ===")
    print("\n--- Baseline Plan ---")
    df_baseline.explain("formatted")
    print("\n--- Optimized Plan ---")
    df_optimized.explain("formatted")

    t0 = time.time()
    base_count = df_baseline.count()
    base_time = time.time() - t0

    t0 = time.time()
    opt_count = df_optimized.count()
    opt_time = time.time() - t0

    assert base_count == opt_count, f"Mismatch! {base_count} vs {opt_count}"
    if base_count < 10000:
        assert (
            df_baseline.orderBy(*df_baseline.columns).collect()
            == df_optimized.orderBy(*df_optimized.columns).collect()
        )

    speedup = ((base_time - opt_time) / base_time) * 100 if base_time > 0 else 0
    print(f"Row Count:      {base_count:,}")
    print(f"Baseline Time:  {base_time:.2f}s")
    print(f"Optimized Time: {opt_time:.2f}s")
    print(f"Speedup:        {speedup:.1f}%\n")


trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
zones = read_delta(spark, SILVER / "taxi_zones")

# --- Cache ---
weather_hourly = weather.groupBy("observation_date", "observation_hour").agg(
    avg("temperature_c").alias("temperature_c")
)
intermediate_df = (
    trips.filter("fare_amount > 0 AND trip_distance > 0")
    .select("pickup_date", "pickup_hour", "fare_amount", "trip_distance")
    .join(
        broadcast(weather_hourly),
        (trips.pickup_date == weather_hourly.observation_date)
        & (trips.pickup_hour == weather_hourly.observation_hour),
        "inner",
    )
)

t0 = time.time()
c1 = intermediate_df.groupBy("pickup_hour").avg("fare_amount").count()
c2 = intermediate_df.groupBy("temperature_c").avg("trip_distance").count()
base_time = time.time() - t0

t0 = time.time()
cached_df = intermediate_df.cache()
cached_df.count()
o1 = cached_df.groupBy("pickup_hour").avg("fare_amount").count()
o2 = cached_df.groupBy("temperature_c").avg("trip_distance").count()
opt_time = time.time() - t0
cached_df.unpersist()
assert c1 == o1 and c2 == o2
print("=== Cache Performance ===")
print(f"Baseline (uncached, 2 queries): {base_time:.2f}s")
print(f"Optimized (cached, 2 queries):  {opt_time:.2f}s")
print(f"Speedup: {((base_time - opt_time) / base_time) * 100:.1f}%\n")

# --- Partition pruning ---
df_base = trips.filter("date(pickup_datetime) = '2024-01-01'")
df_opt = trips.filter(
    "pickup_date = '2024-01-01' AND date(pickup_datetime) = '2024-01-01'"
)
benchmark_and_verify(df_base, df_opt, "Partition Pruning")

# --- Broadcast join ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
df_base = trips.join(zones, trips.pickup_location_id == zones.location_id)
df_opt = trips.join(broadcast(zones), trips.pickup_location_id == zones.location_id)
benchmark_and_verify(df_base, df_opt, "Broadcast Join")

# --- AQE ---
query_df = trips.groupBy("pickup_location_id", "pickup_hour").agg(
    {"fare_amount": "avg", "trip_distance": "sum"}
)
spark.conf.set("spark.sql.adaptive.enabled", "false")
t0 = time.time()
n_off = query_df.count()
t_off = time.time() - t0

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
t0 = time.time()
n_on = query_df.count()
t_on = time.time() - t0
assert n_off == n_on
print("=== AQE Performance ===")
print(f"AQE Disabled: {t_off:.2f}s")
print(f"AQE Enabled:  {t_on:.2f}s")
print(f"Speedup:      {((t_off - t_on) / t_off) * 100:.1f}%\n")

=== Cache Performance ===
Baseline (uncached, 2 queries): 2.69s
Optimized (cached, 2 queries):  2.81s
Speedup: -4.4%

=== Partition Pruning ===

--- Baseline Plan ---
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [14]: [taxi_type#42612, vendor_id#42613, pickup_datetime#42614, dropoff_datetime#42615, passenger_count#42616, trip_distance#42617, pickup_location_id#42618, dropoff_location_id#42619, fare_amount#42620, tip_amount#42621, tolls_amount#42622, total_amount#42623, pickup_hour#42625, pickup_date#42624]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
PushedFilters: [IsNotNull(pickup_datetime), GreaterThanOrEqual(pickup_datetime,2024-01-01 01:00:00.0), LessThan(pickup_datetime,2024-01-02 01:00:00.0)]
ReadSchema: struct<taxi_type:string,vendor_id:int,pickup_datetime:timestamp,dropoff_datetime:timestamp,passe

## 5. Gold data products

Same builders as `data_products.ipynb`.

In [6]:
from src.gold.products import build_products

print("=== GOLD data products ===")
build_products(spark, force=FORCE_PRODUCTS)

=== GOLD data products ===
--- Pipeline Execution Plan (Schema Version: 1.0) ---
[REFRESHING] Product: daily_borough_mobility...


Product daily_borough_mobility refreshed successfully.
[REFRESHING] Product: taxi_zone_monthly_demand...


Product taxi_zone_monthly_demand refreshed successfully.
[REFRESHING] Product: weather_impact_summary...
Product weather_impact_summary refreshed successfully.
[REFRESHING] Product: air_quality_demand_summary...


Product air_quality_demand_summary refreshed successfully.
[REFRESHING] Product: zone_weather_sensitivity...


Product zone_weather_sensitivity refreshed successfully.


## 6. Benchmarks (products vs on-demand + AQE)

Same evaluations as `benchmark.ipynb`.

In [7]:
from src.benchmark.evaluate import evaluate_query
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND as query_1,
    QUERY_2_WEATHER_DISTANCE as query_2,
    QUERY_3_PM25_DEMAND as query_3,
    QUERY_4_ZONE_WEATHER_SENSITIVITY as query_4,
    QUERY_5_PEAK_HOURS_BY_DOW as query_5,
    QUERY_6_MONTHLY_TRENDS as query_6,
)

read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)
prod = GOLD / "data_products"

evaluate_query(
    spark,
    "Query 1: Monthly Taxi Demand per Zone",
    spark.sql(query_1),
    read_delta(spark, prod / "taxi_zone_monthly_demand").select(
        "trip_month", "pickup_location_id", "pickup_borough", "pickup_zone",
        "total_trips", "active_days", "avg_daily_trips",
    ),
    prod / "taxi_zone_monthly_demand",
)
evaluate_query(
    spark,
    "Query 2: Average Distance by Weather",
    spark.sql(query_2),
    read_delta(spark, prod / "weather_impact_summary").select(
        "temp_category", "wind_category", "trip_count", "avg_distance_miles"
    ),
    prod / "weather_impact_summary",
)
evaluate_query(
    spark,
    "Query 3: Air Quality vs Demand",
    spark.sql(query_3),
    read_delta(spark, prod / "air_quality_demand_summary").select(
        "pm25_level", "trips", "observed_hours", "trips_per_hour"
    ),
    prod / "air_quality_demand_summary",
)
evaluate_query(
    spark,
    "Query 4: Zone Weather Demand Variation",
    spark.sql(query_4),
    read_delta(spark, prod / "zone_weather_sensitivity").select(
        "pickup_zone", "coldest", "cool", "warm", "warmest", "pct_variation"
    ),
    prod / "zone_weather_sensitivity",
)
evaluate_query(spark, "Query 5: Peak Travel Hours by Day of Week", aqe_query=query_5)
evaluate_query(spark, "Query 6: Monthly Trends in Taxi Demand", aqe_query=query_6)


EVALUATING: Query 1: Monthly Taxi Demand per Zone

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [pickup_location_id#56026, pickup_zone#56027, pickup_borough#56028, pickup_date#56040]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#56040)]
PushedFilters: [IsNotNull(pickup_zone), Not(EqualTo(pickup_zone,UNKNOWN))]
ReadSchema: struct<pickup_location_id:int,pickup_zone:string,pickup_borough:string>

(2) Filter
Input [4]:


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       12
Baseline Time:     0.48s
Optimized Time:    0.16s
Speedup:           66.5%
Storage Overhead:  6.6 KB (2 files)


EVALUATING: Query 3: Air Quality vs Demand

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pm25#56038, pickup_hour#56041, pickup_date#56040]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#5


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       33
Baseline Time:     0.76s
Optimized Time:    0.14s
Speedup:           82.2%
Storage Overhead:  6.6 KB (2 files)


EVALUATING: Query 4: Zone Weather Demand Variation

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- Project (17)
         +- Filter (16)
            +- HashAggregate (15)
               +- HashAggregate (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- HashAggregate (11)
                           +- HashAggregate (10)
                              +- HashAggregate (9)
                                 +- HashAggregate (8)
                                    +- Project (7)
                                       +- Window (6)
                                          +- Sort (5)
                                       


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       53
Baseline Time:     3.00s
Optimized Time:    0.21s
Speedup:           93.1%
Storage Overhead:  10.7 KB (2 files)


EVALUATING: Query 5: Peak Travel Hours by Day of Week

--- Baseline Physical Plan (AQE Disabled) ---
== Physical Plan ==
* Project (21)
+- * Sort (20)
   +- Exchange (19)
      +- * Project (18)
         +- * Filter (17)
            +- Window (16)
               +- WindowGroupLimit (15)
                  +- * Sort (14)
                     +- Exchange (13)
                        +- WindowGroupLimit (12)
                           +- * Sort (11)
                              +- * HashAggregate (10)
                                 +- Exchange (9)
                                    +- * HashAggregate (8)
                                       +- * HashAggregate (7)
                                          +- Exchange (6)
                                   

26/09/24 18:10:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



--- Optimized Physical Plan (AQE Enabled) ---
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- Window (11)
      +- Sort (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Exchange (7)
                  +- HashAggregate (6)
                     +- HashAggregate (5)
                        +- Exchange (4)
                           +- HashAggregate (3)
                              +- Project (2)
                                 +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [pickup_date#56040]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#56040)]
ReadSchema: struct<>

(2) Project
Output [2]: [pickup_date#56040, trunc(pickup_date#56040, MM) AS _groupingexpression#63523]
Input [1]: [pickup_date#56040]

(3) HashAggregate
Input [2]: [pickup_date#56040, _groupingexpression#63523]
Keys [2]: [_gr

26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 1


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       4
Baseline Time:     0.31s
Optimized Time:    0.24s
Speedup:           24.2%
Storage Overhead:  N/A (AQE Engine Optimization on Integrated Table)



26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 18:10:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


{'query': 'Query 6: Monthly Trends in Taxi Demand',
 'rows': 4,
 'baseline_s': 0.3148689270019531,
 'optimized_s': 0.2386617660522461,
 'speedup_pct': 24.202820416520275,
 'storage': 'N/A (AQE Engine Optimization on Integrated Table)'}

## 7. Incremental data + schema evolution re-ingest

Generate update files (`data_generator.ipynb`), then re-run bronze → silver → gold.
Extra columns (`humidity`, `aqi`) are allowed by schema validation and flow through silver/gold when present — no YAML rewrite required.

In [ ]:
from pyspark.sql import SparkSession
from src.generators.incremental import run_all_generators
from src.spark import create_spark

if RUN_INCREMENTAL:
    # Recover if an earlier generator call stopped the session
    if SparkSession.getActiveSession() is None:
        print("Spark session was stopped; recreating...")
        spark = create_spark("run-pipeline")

    print("=== Generate incremental updates ===")
    run_all_generators(spark=spark)

    print("\n=== Re-ingest with updates (schema evolution) ===")
    ingest_all(spark, ["taxi_trips", "weather", "air_quality"])
    promote_all(spark, ["taxi_trips", "weather", "air_quality"])

    weather_s = read_delta(spark, SILVER / "weather")
    aq_s = read_delta(spark, SILVER / "air_quality")
    print("silver weather columns:", weather_s.columns)
    print("silver air_quality columns:", aq_s.columns)
    print("humidity present:", "humidity" in weather_s.columns)
    print("aqi present:", "aqi" in aq_s.columns)

    print("\n=== Re-integrate + refresh products ===")
    integrate(spark)
    integrated = read_delta(spark, GOLD / "integrated_taxi_trips")
    print("gold integrated columns:", integrated.columns)
    build_products(spark, force=True)
else:
    print("Skipped incremental demo (RUN_INCREMENTAL=False)")


## 8. Monitoring report

Ops metrics appended by every bronze/silver/gold step into `data/lake/ops/pipeline_runs`.

In [9]:
from src.monitoring.queries import print_monitoring_report
from src.monitoring.recorder import PIPELINE_RUNS

print("ops table:", PIPELINE_RUNS)
print_monitoring_report(spark)

spark.stop()
print("\nFull pipeline demo finished.")

ops table: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/ops/pipeline_runs


26/09/24 18:12:14 WARN DeltaLog: Failed to parse file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/ops/pipeline_runs/_delta_log/_last_checkpoint. This may happen if there was an error during read operation, or a file appears to be partial. Sleeping and trying again.
java.lang.NullPointerException: Cannot invoke "org.apache.spark.SparkEnv.conf()" because the return value of "org.apache.spark.SparkEnv$.get()" is null
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.<init>(DelegatingLogStore.scala:38)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStoreWithClassName(LogStore.scala:288)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:385)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:380)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStore(LogStore.scala:266)
	at org.apache.spark.sql.delta.storage.LogStore$.apply(LogStore.scala:279)
	at org.apache.spark.s

Py4JJavaError: An error occurred while calling o1662.load.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.SparkEnv.conf()" because the return value of "org.apache.spark.SparkEnv$.get()" is null
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.<init>(DelegatingLogStore.scala:38)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStoreWithClassName(LogStore.scala:288)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:385)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:380)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStore(LogStore.scala:266)
	at org.apache.spark.sql.delta.storage.LogStore$.apply(LogStore.scala:279)
	at org.apache.spark.sql.delta.storage.LogStore$.apply(LogStore.scala:274)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:322)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:3